# Image Classification Robustness and Efficiency Benchmark
This notebook orchestrates the reusable package. A smoke test checks integration only; it is not a final experiment. Run cells in order and keep the attached Intel dataset read-only under `/kaggle/input`.

## 1. Environment and GPU verification

In [ ]:
import platform, sys
from pathlib import Path
import torch, torchvision, numpy, pandas, sklearn, matplotlib
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
print('Kaggle inputs:', [str(path) for path in Path('/kaggle/input').iterdir()])

## 2. Dependency installation (only when required)
Set `PROJECT_ROOT` to the uploaded/cloned repository. The editable install registers local code with `--no-deps`, so it does not upgrade Kaggle packages or require internet.

In [ ]:
PROJECT_ROOT = Path('/kaggle/working/image-robustness')  # EDIT if needed
assert (PROJECT_ROOT / 'pyproject.toml').is_file(), f'Wrong PROJECT_ROOT: {PROJECT_ROOT}'
%pip install -e {PROJECT_ROOT} --no-deps -q

## 3. Dataset discovery and path configuration
Edit `DATASET_ROOT` after inspecting the paths printed above. `SMOKE_TEST=True` limits epochs and batches. The test split is never used for training or selection.

## 4. Project import and experiment setup
The next cell loads the package, resolves configuration, creates only writable output directories, and prints the discovered data contract before any training.

In [ ]:
from dataclasses import asdict
from robustness.config import load_config
from robustness.data import build_dataloaders, format_data_summary
from robustness.utils import prepare_experiment, save_json_atomic
DATASET_ROOT = '/kaggle/input/intel-image-classification'  # EDIT after inspection
OUTPUT_DIR = '/kaggle/working/image-robustness-results'
SMOKE_TEST = True
SKIP_TRAINING = False
overrides = {'data.dataset_root': DATASET_ROOT, 'output_dir': OUTPUT_DIR}
if SMOKE_TEST:
    overrides.update({'training.epochs': 1, 'training.max_train_batches': 10, 'training.max_validation_batches': 3, 'benchmark.warmup_iterations': 2, 'benchmark.timed_iterations': 5})
cfg = load_config(PROJECT_ROOT / 'configs/default.yaml', overrides)
device, output_root = prepare_experiment(cfg)
bundle = build_dataloaders(cfg)
print(format_data_summary(bundle.summary))
images, labels = next(iter(bundle.train))
print('Batch:', images.shape, labels.shape, images.dtype, 'device:', device)
save_json_atomic(asdict(cfg), output_root / 'resolved_config.json')

## 5. Baseline CNN training
The baseline learns all features from scratch. The shared training cell below trains it before moving to MobileNetV2.

## 6. MobileNetV2 training
MobileNetV2 initially freezes its pretrained feature extractor and trains the replaced six-class head. The best validation checkpoint—not the last epoch—is reloaded. Pretrained weights can require internet on first use.

In [ ]:
from robustness.models import build_model, count_parameters
from robustness.train import train_model, load_checkpoint
models = {}
checkpoint_paths = {}
for model_name in ('baseline', 'mobilenet_v2'):
    checkpoint = output_root / model_name / 'best_model.pt'
    model = build_model(model_name, cfg, len(bundle.class_names), pretrained_override=False if SKIP_TRAINING else None)
    print(model_name, 'total/trainable:', count_parameters(model), count_parameters(model, True))
    if not SKIP_TRAINING:
        result = train_model(model, bundle.train, bundle.val, cfg, device, output_root / model_name)
        checkpoint = result.checkpoint_path
    load_checkpoint(model, checkpoint, device)
    models[model_name] = model
    checkpoint_paths[model_name] = checkpoint
if SMOKE_TEST: print('SMOKE TEST ONLY — do not report these metrics as final results.')

## 7. Clean test evaluation

In [ ]:
from robustness.evaluate import evaluate_model
evaluation_results = []
for model_name, model in models.items():
    result = evaluate_model(model, bundle.test, bundle.class_names, device, model_name, 'clean', output_root / 'metrics')
    evaluation_results.append(result)
    print(model_name, 'clean accuracy/macro-F1:', result.accuracy, result.macro_f1)

## 8. Degradation experiments

In [ ]:
from robustness.degradations import build_degradations
from robustness.data import build_degraded_test_loader
conditions = build_degradations(cfg.degradations)
for condition, degradation in conditions.items():
    if condition == 'clean': continue
    loader = build_degraded_test_loader(bundle, degradation, cfg)
    for model_name, model in models.items():
        result = evaluate_model(model, loader, bundle.class_names, device, model_name, condition, output_root / 'metrics')
        evaluation_results.append(result)
        print(model_name, condition, result.accuracy, result.macro_f1)

## 9. CPU/GPU latency benchmarking

In [ ]:
from robustness.benchmark import benchmark_model, save_benchmark
clean_batch, _ = next(iter(bundle.test))  # decoding and transforms finish before timing
clean_batch = clean_batch[:cfg.benchmark.batch_size]
benchmarks = []
for model_name, model in models.items():
    devices = [torch.device('cpu')] + ([torch.device('cuda')] if torch.cuda.is_available() else [])
    for benchmark_device in devices:
        result = benchmark_model(model, clean_batch, benchmark_device, model_name, cfg.benchmark.warmup_iterations, cfg.benchmark.timed_iterations, checkpoint_paths[model_name])
        benchmarks.append(result)
        save_benchmark(result, output_root / 'metrics/benchmarks' / f'{model_name}_{benchmark_device.type}.json')
        print(result)

## 10. Failure analysis

In [ ]:
from robustness.error_analysis import collect_misclassifications, save_error_gallery
for condition in cfg.evaluation.error_conditions:
    loader = build_degraded_test_loader(bundle, conditions[condition], cfg)
    for model_name, model in models.items():
        errors = collect_misclassifications(model, loader, device, cfg.evaluation.error_examples)
        save_error_gallery(errors, bundle.class_names, model_name, condition, output_root / 'figures/errors' / model_name / condition)
        print(model_name, condition, 'saved errors:', len(errors))

## 11. Results summary and export
Only interpret these outputs after a full run with `SMOKE_TEST=False`. Observations should quote measured values; causal explanations remain hypotheses unless tested independently.

In [ ]:
from robustness.evaluate import save_evaluation_summary, plot_metric_comparison
summary_path = save_evaluation_summary(evaluation_results, output_root / 'metrics/evaluation_summary.csv')
plot_metric_comparison(evaluation_results, output_root / 'figures/accuracy_comparison.png', 'accuracy')
plot_metric_comparison(evaluation_results, output_root / 'figures/macro_f1_comparison.png', 'macro_f1')
summary = pandas.read_csv(summary_path)
display(summary)
print('Artifacts saved under:', output_root)
if SMOKE_TEST: print('Do not interpret or publish this smoke-test table.')